# Results Comparison — Baseline vs Deep Models
Consolidates all experiment outputs into a single comparison report.
Loads saved CSVs and JSONs — no re-training required.

In [ ]:
import sys, os, json
sys.path.insert(0, ".")
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    GITHUB_REPO = "https://github.com/YOUR_USERNAME/emotion-recognition.git"
    !git clone {GITHUB_REPO} /content/emotion-recognition
    %cd /content/emotion-recognition

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

sns.set_theme(style="darkgrid")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})
RESULTS_DIR = "outputs/results"
TARGETS     = ["valence", "arousal", "dominance"]
COLORS      = {"baseline": "#4C72B0", "deep": "#DD8452",
               "within"  : "#55A868", "loso": "#C44E52"}

In [ ]:
# ── Baseline results ───────────────────────────────────────────────
baseline_path = os.path.join(RESULTS_DIR, "baseline_results.csv")
df_baseline   = pd.read_csv(baseline_path) if os.path.exists(baseline_path) \
                else pd.DataFrame()
print(f"Baseline rows: {len(df_baseline)}")

# ── Deep LOSO results ──────────────────────────────────────────────
deep_rows = []
for target in TARGETS:
    for model_type in ["fusion", "cnnlstm", "cnn", "eegnet"]:
        p = os.path.join(RESULTS_DIR,
                         f"loso_deep_{model_type}_{target}.json")
        if os.path.exists(p):
            with open(p) as f:
                d = json.load(f)
            deep_rows.append({
                "target"  : target,
                "model"   : model_type,
                "eval"    : "loso",
                "acc_mean": d["acc_mean"],
                "acc_std" : d["acc_std"],
                "f1_mean" : d["f1_mean"],
                "f1_std"  : d["f1_std"],
                "auc_mean": d["auc_mean"],
                "auc_std" : d["auc_std"],
            })

df_deep = pd.DataFrame(deep_rows)
print(f"Deep LOSO rows: {len(df_deep)}")

# ── Optuna best params (if available) ─────────────────────────────
optuna_results = {}
for target in TARGETS:
    p = os.path.join(RESULTS_DIR, f"optuna_fusion_{target}.json")
    if os.path.exists(p):
        with open(p) as f:
            optuna_results[target] = json.load(f)

print(f"Optuna results loaded: {list(optuna_results.keys())}")

## 1. Head-to-Head: Baseline vs Deep — LOSO AUC

In [ ]:
if df_baseline.empty or df_deep.empty:
    print("⚠ Missing result files — run training notebooks first")
else:
    # Best baseline per target (LOSO)
    bl_loso = df_baseline[df_baseline["eval"] == "loso"].copy()
    bl_best = (bl_loso.sort_values("auc_mean", ascending=False)
                       .groupby("target").first().reset_index())
    bl_best["model_class"] = "Baseline (" + bl_best["model"].str.upper() + ")"

    # Best deep per target
    dp_best = (df_deep.sort_values("auc_mean", ascending=False)
                      .groupby("target").first().reset_index())
    dp_best["model_class"] = "Deep (" + dp_best["model"].str.upper() + ")"

    combined = pd.concat([bl_best, dp_best], ignore_index=True)

    fig, axes = plt.subplots(1, 3, figsize=(16, 6), sharey=False)
    metrics   = [("auc_mean", "auc_std", "ROC-AUC"),
                 ("f1_mean",  "f1_std",  "F1 Score"),
                 ("acc_mean", "acc_std", "Accuracy")]

    for ax, (m_col, s_col, m_label) in zip(axes, metrics):
        for target in TARGETS:
            sub = combined[combined["target"] == target]
            x   = range(len(sub))
            ax.bar(
                [t + TARGETS.index(target) * (len(sub) + 1) for t in x],
                sub[m_col].values,
                yerr=sub[s_col].values,
                capsize=4,
                color=[COLORS["baseline"], COLORS["deep"]],
                edgecolor="white", alpha=0.88, width=0.7,
            )

        tick_pos = [
            i * (2 + 1) + 0.5
            for i in range(len(TARGETS))
        ]
        ax.set_xticks(tick_pos)
        ax.set_xticklabels(
            [f"{t.capitalize()}\nBaseline vs Deep"
             for t in TARGETS], fontsize=8
        )
        ax.set_title(m_label, fontweight="bold")
        ax.set_ylabel(m_label)
        ax.set_ylim(0.4, 1.0)
        ax.axhline(0.5, color="black", linestyle="--",
                   linewidth=0.8, alpha=0.5, label="Chance")
        ax.legend(fontsize=7)

    plt.suptitle(
        "Baseline vs Deep Model — LOSO Performance",
        fontsize=14, fontweight="bold"
    )
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/comparison_baseline_vs_deep.png",
                bbox_inches="tight")
    plt.show()

## 2. Full Results Table

In [ ]:
all_rows = []

if not df_baseline.empty:
    for _, r in df_baseline.iterrows():
        all_rows.append({
            "Type"   : "Baseline",
            "Model"  : r["model"].upper(),
            "Target" : r["target"].capitalize(),
            "Protocol": r["eval"].upper(),
            "Acc"    : f"{r['acc_mean']:.3f} ± {r['acc_std']:.3f}",
            "F1"     : f"{r['f1_mean']:.3f}  ± {r['f1_std']:.3f}",
            "AUC"    : f"{r['auc_mean']:.3f} ± {r['auc_std']:.3f}",
        })

if not df_deep.empty:
    for _, r in df_deep.iterrows():
        all_rows.append({
            "Type"   : "Deep",
            "Model"  : r["model"].upper(),
            "Target" : r["target"].capitalize(),
            "Protocol": "LOSO",
            "Acc"    : f"{r['acc_mean']:.3f} ± {r['acc_std']:.3f}",
            "F1"     : f"{r['f1_mean']:.3f}  ± {r['f1_std']:.3f}",
            "AUC"    : f"{r['auc_mean']:.3f} ± {r['auc_std']:.3f}",
        })

df_all = pd.DataFrame(all_rows)
print(df_all.to_string(index=False))
df_all.to_csv(f"{RESULTS_DIR}/full_comparison_table.csv", index=False)
print(f"\n✅ Saved: {RESULTS_DIR}/full_comparison_table.csv")

## 3. Per-Target Deep LOSO Breakdown

In [ ]:
if not df_deep.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    pal = sns.color_palette("muted", len(df_deep["model"].unique()))

    for ax, target in zip(axes, TARGETS):
        sub = df_deep[df_deep["target"] == target].copy()
        sub = sub.sort_values("auc_mean", ascending=True)
        ax.barh(
            sub["model"].str.upper(),
            sub["auc_mean"],
            xerr=sub["auc_std"],
            color=pal[:len(sub)],
            edgecolor="white",
            capsize=4,
        )
        ax.axvline(0.5, color="black", linestyle="--",
                   linewidth=0.8, alpha=0.5)
        ax.set_xlim(0.3, 1.0)
        ax.set_title(target.capitalize(), fontweight="bold")
        ax.set_xlabel("ROC-AUC (LOSO)")

    plt.suptitle("Deep Model LOSO AUC by Target",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/deep_model_comparison.png",
                bbox_inches="tight")
    plt.show()

## 4. Optuna Best Hyperparameters Summary

In [ ]:
if optuna_results:
    rows = []
    for target, res in optuna_results.items():
        row = {"target": target, "best_auc": res["best_auc"]}
        row.update(res["best_params"])
        rows.append(row)
    df_optuna = pd.DataFrame(rows)
    print("── Optuna Best Hyperparameters ──")
    print(df_optuna.to_string(index=False))
    df_optuna.to_csv(f"{RESULTS_DIR}/optuna_best_params.csv", index=False)
else:
    print("No Optuna results found — run tuner.py first")

In [ ]:
import json

summary = {
    "baseline_results": df_baseline.to_dict(orient="records")
        if not df_baseline.empty else [],
    "deep_results"    : df_deep.to_dict(orient="records")
        if not df_deep.empty else [],
    "optuna_results"  : optuna_results,
}
with open(f"{RESULTS_DIR}/master_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

if IN_COLAB:
    !git config --global user.email "you@example.com"
    !git config --global user.name "Your Name"
    !git add outputs/results/ notebooks/
    !git commit -m "feat: full results comparison notebook"
    !git push
    print("✅ Pushed to GitHub")